In [1]:
# Cell 1: imports, paths, basic config (no cfg usage yet)

import json
import time
from dataclasses import dataclass, asdict
from collections import Counter, defaultdict
from pathlib import Path
from typing import List, Dict, Tuple, Optional
import requests
import pandas as pd
import numpy as np

import openai
import anthropic
import google.generativeai as genai

BASE_DIR = Path(r"C:\Users\STSI\OneDrive - Skagerak Energi\06-NæringsPhD\Egne papers\State of the art")
DATA_DIR = BASE_DIR / "Data"
RESULTS_DIR = BASE_DIR / "softwareanalysis_refactored"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SOFTWARE_CSV = DATA_DIR / "methods_osmm_update.csv"
METHOD_VARIANTS_JSON = DATA_DIR / "variantgroups.json"

ASSESS_RAW_JSON = RESULTS_DIR / "assessments_raw.json"
ASSESS_AGG_JSON = RESULTS_DIR / "assessments_agg.json"
MATRIX_CSV = RESULTS_DIR / "software_methods_matrix.csv"

BATCH_SIZE = 20
MAX_RETRIES = 3
TIMEOUT = 180


c:\git_repos\Literature-search-and-analysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Cell 2: API init, CreditTracker, Zotero config

import configparser
import tiktoken

def load_llm_config(path: Path = Path("config_LLM.txt")) -> configparser.ConfigParser:
    cfg = configparser.ConfigParser()
    cfg.read(path)
    return cfg

def initialize_openai(cfg: configparser.ConfigParser):
    key = cfg["LLM"].get("OPENAI_API_KEY", "")
    model = cfg["LLM"].get("MODELTYPE_ADV", "gpt-4o-mini")
    if not key:
        return None, None
    client = openai.OpenAI(api_key=key)
    return client, model

def initialize_anthropic(cfg: configparser.ConfigParser):
    key = cfg["LLM"].get("ANTHROPIC_API_KEY", "")
    if not key:
        return None
    client = anthropic.Anthropic(api_key=key)
    return client

def initialize_google(cfg: configparser.ConfigParser):
    key = cfg["LLM"].get("GOOGLE_API_KEY", "")
    if not key:
        return False
    genai.configure(api_key=key)
    return True

def initialize_perplexity(cfg: configparser.ConfigParser):
    key = cfg["LLM"].get("PERPLEXITY_API_KEY", "")
    if not key:
        return None
    client = openai.OpenAI(api_key=key, base_url="https://api.perplexity.ai")
    return client

def num_tokens_from_string(s: str, model_name: str) -> int:
    try:
        enc = tiktoken.encoding_for_model(model_name)
        return len(enc.encode(s))
    except KeyError:
        if model_name.startswith("gpt-4"):
            enc = tiktoken.get_encoding("cl100k_base")
            return len(enc.encode(s))
        else:
            return len(s) // 4  # rough fallback

def count_tokens_in_messages(messages: List[Dict], model: str) -> int:
    total = 0
    for m in messages:
        content = m.get("content", "")
        if isinstance(content, str):
            total += num_tokens_from_string(content, model)
    total += 4  # overhead
    return total

class CreditTracker:
    PRICING = {
        # OpenAI
        "gpt-4o": {"input": 1.25, "output": 5.00},
        "gpt-4o-mini": {"input": 0.075, "output": 0.30},
        # Claude
        "claude-3-haiku-20240307": {"input": 0.25, "output": 1.25},
        "claude-3-5-haiku-20241022": {"input": 0.80, "output": 4.00},
        "claude-3-5-sonnet-20241022": {"input": 3.00, "output": 15.00},
        "claude-sonnet-4-20250514": {"input": 3.00, "output": 15.00},
        # Gemini
        "models/gemini-2.5-flash": {"input": 0.075, "output": 0.30},
        "models/gemini-2.0-flash": {"input": 0.075, "output": 0.30},
    }

    def __init__(self):
        self.total_input_tokens = 0
        self.total_output_tokens = 0
        self.total_cached_tokens = 0
        self.total_cost = 0.0
        self.model_usage: Dict[str, Dict] = {}
        self.call_count = 0

    def update(self, model: str, input_tokens: int, output_tokens: int, cached_tokens: int = 0):
        self.total_input_tokens += input_tokens
        self.total_output_tokens += output_tokens
        self.total_cached_tokens += cached_tokens
        self.call_count += 1

        pricing = self.PRICING.get(model, {"input": 0.00015, "output": 0.0006})
        in_cost = input_tokens / 1_000_000 * pricing["input"]
        out_cost = output_tokens / 1_000_000 * pricing["output"]
        call_cost = in_cost + out_cost
        self.total_cost += call_cost

        if model not in self.model_usage:
            self.model_usage[model] = {"calls": 0, "input_tokens": 0,
                                       "output_tokens": 0, "cached_tokens": 0,
                                       "cost": 0.0}
        m = self.model_usage[model]
        m["calls"] += 1
        m["input_tokens"] += input_tokens
        m["output_tokens"] += output_tokens
        m["cached_tokens"] += cached_tokens
        m["cost"] += call_cost

    def get_summary_text(self) -> str:
        total_tokens = self.total_input_tokens + self.total_output_tokens
        avg = self.total_cost / max(self.call_count, 1)
        lines = []
        lines.append("API USAGE SUMMARY")
        lines.append(f"Total calls: {self.call_count}")
        lines.append(f"Total tokens: {total_tokens} "
                     f"(input {self.total_input_tokens}, output {self.total_output_tokens}, "
                     f"cached {self.total_cached_tokens})")
        lines.append(f"Total cost: {self.total_cost:.4f} USD")
        lines.append(f"Average cost per call: {avg:.4f} USD")
        if self.model_usage:
            lines.append("By model:")
            for model, stats in self.model_usage.items():
                tot = stats["input_tokens"] + stats["output_tokens"]
                lines.append(f"  {model}: calls {stats['calls']}, "
                             f"tokens {tot}, cost {stats['cost']:.4f}")
        return "\n".join(lines)

cfg = load_llm_config()
openai_client, default_openai_model = initialize_openai(cfg)
anthropic_client = initialize_anthropic(cfg)
google_enabled = initialize_google(cfg)
perplexity_client = initialize_perplexity(cfg)
credit_tracker = CreditTracker()

# Zotero specification (group library)
ZOTERO_API_KEY = cfg["LLM"].get("ZOTERO_API_KEY", "")
ZOTERO_GROUP_ID = cfg["LLM"].get("ZOTERO_GROUP_ID", "")
ZOTERO_COLLECTION_KEY = cfg["LLM"].get("ZOTERO_COLLECTION_KEY", "")

if not (ZOTERO_API_KEY and ZOTERO_GROUP_ID and ZOTERO_COLLECTION_KEY):
    print("Warning: Zotero config incomplete; Zotero-mode assessments will have no refs.")

ZOTERO_API_ROOT = f"https://api.zotero.org/groups/{ZOTERO_GROUP_ID}"


In [3]:
# helper for zotero

def fetch_collection_items(collection_key: str,
                           api_key: str,
                           limit: int = 100) -> List[Dict]:
    """Fetch all items in a Zotero collection via Web API (read-only)."""
    headers = {"Zotero-API-Key": api_key}
    items: List[Dict] = []
    start = 0

    while True:
        params = {
            "format": "json",
            "limit": limit,
            "start": start,
        }
        url = f"{ZOTERO_API_ROOT}/collections/{collection_key}/items"
        r = requests.get(url, headers=headers, params=params)
        r.raise_for_status()
        batch = r.json()
        if not batch:
            break
        items.extend(batch)
        start += len(batch)
    return items

def export_collection_mapping(collection_key: str,
                              api_key: str,
                              outfile: Path) -> None:
    """
    Export raw Zotero items with parsed software/method tags:

    [
      {
        "software": "PSS/E" or "",
        "method": "optimal power flow" or "",
        "title": "...",
        "year": "2020",
        "url": "..."
      },
      ...
    ]
    """
    raw_items = fetch_collection_items(collection_key, api_key)
    mapped: List[Dict] = []

    print(f"Fetched {len(raw_items)} items from Zotero API")

    for it in raw_items:
        data = it.get("data", {})
        title = data.get("title", "")
        year = data.get("date", "")[:4]
        url = data.get("url") or data.get("DOI") or ""

        tags = [t.get("tag", "") for t in data.get("tags", [])]
        sw = next((t.split("software:", 1)[1].strip()
                   for t in tags if t.lower().startswith("software:")), "")
        meth = next((t.split("method:", 1)[1].strip()
                     for t in tags if t.lower().startswith("method:")), "")

        if not sw and not meth:
            continue  # skip completely untagged items

        mapped.append({
            "software": sw,
            "method": meth,
            "title": title,
            "year": year,
            "url": url,
        })

    outfile.parent.mkdir(parents=True, exist_ok=True)
    with open(outfile, "w", encoding="utf-8") as f:
        json.dump(mapped, f, indent=2, ensure_ascii=False)

    print(f"Exported {len(mapped)} tagged items to {outfile}")



In [4]:
#run whenever new docs are added to the collection...
ZOTERO_MAPPING_FILE = DATA_DIR / "zotero_pair_refs.json"
if ZOTERO_API_KEY and ZOTERO_COLLECTION_KEY:
    export_collection_mapping(ZOTERO_COLLECTION_KEY, ZOTERO_API_KEY, ZOTERO_MAPPING_FILE)
else:
    print("Zotero API key or collection key missing; skipping export.")

Fetched 145 items from Zotero API
Exported 128 tagged items to C:\Users\STSI\OneDrive - Skagerak Energi\06-NæringsPhD\Egne papers\State of the art\Data\zotero_pair_refs.json


In [ ]:
# Cell 3: Data classes and assessor (batch, Zotero vs web modes)

@dataclass
class AssessmentResult:
    software: str
    method: str
    rank: int
    reasoning: str
    sources: List[str]
    llm_provider: str
    llm_model: str
    mode: str            # "zotero" or "web"
    batch_id: str
    input_tokens: int = 0
    output_tokens: int = 0

@dataclass
class ConsensusResult:
    software: str
    method: str
    final_rank: int
    confidence: float
    agreement_level: str
    individual_ranks: Dict[str, int]
    individual_modes: Dict[str, str]
    individual_sources: Dict[str, List[str]]
    total_assessments: int

class SoftwareMethodAssessor:
    """
    Orchestrates:
    - batch assessments,
    - multiple models,
    - zotero-only vs web-search assessments,
    - consensus aggregation.
    """

    def __init__(self,
                 system_prompt_zotero: str,
                 system_prompt_web: str,
                 timeout: int = TIMEOUT,
                 max_retries: int = MAX_RETRIES):
        self.timeout = timeout
        self.max_retries = max_retries
        self.system_prompt_zotero = system_prompt_zotero
        self.system_prompt_web = system_prompt_web

    @staticmethod
    def calc_confidence(ranks: List[int]) -> Tuple[float, str]:
        if not ranks:
            return 0.0, "no-data"
        counts = Counter(ranks)
        most_common_rank, most_common_count = counts.most_common(1)[0]
        total = len(ranks)
        conf = most_common_count / total
        if total == 1:
            level = "single-assessment"
        elif conf == 1.0:
            level = "perfect-agreement"
        elif conf >= 0.75:
            level = "strong-agreement"
        elif conf >= 0.5:
            level = "moderate-agreement"
        else:
            level = "weak-agreement"
        return conf, level

    def build_batch_prompt(self,
                        items: List[Tuple[str, str]],
                        mode: str,
                        zotero_refs: Optional[Dict[str, Dict]] = None
                        ) -> str:
        """
        mode: "zotero" or "web"

        zotero_refs must have keys:
        - "by_software"
        - "by_method"
        - "by_pair"
        """
        if mode == "zotero":
            sys = self.system_prompt_zotero
        else:
            sys = self.system_prompt_web

        lines = []
        lines.append(sys)
        lines.append("")
        lines.append(f"You must assess {len(items)} software–method pairs independently.")
        lines.append("Return ONLY a JSON array with one object per item.")
        lines.append("Each object: {software, method, rank, reasoning, sources}.")

        by_sw = zotero_refs.get("by_software", {}) if zotero_refs else {}
        by_m = zotero_refs.get("by_method", {}) if zotero_refs else {}
        by_pair = zotero_refs.get("by_pair", {}) if zotero_refs else {}

        for idx, (sw, meth) in enumerate(items, 1):
            lines.append("")
            lines.append(f"{idx}. Software: {sw}")
            lines.append(f"   Method: {meth}")

            if mode == "zotero" and zotero_refs is not None:
                refs = []

                # Most specific: items tagged with both this software and method
                refs.extend(by_pair.get((sw, meth), []))
                # Software-only items
                refs.extend(by_sw.get(sw, []))
                # Method-only items
                refs.extend(by_m.get(meth, []))

                # Deduplicate by (title, url)
                seen = set()
                uniq = []
                for r in refs:
                    key = (r.get("title", ""), r.get("url", ""))
                    if key in seen:
                        continue
                    seen.add(key)
                    uniq.append(r)

                if uniq:
                    lines.append("   References (Zotero subset for this pair):")
                    for r in uniq:
                        title = r.get("title", "")
                        year = r.get("year", "")
                        url = r.get("url", "")
                        lines.append(f"   - {title} ({year}) - {url}")

        lines.append("")
        lines.append("Important:")
        lines.append("- In 'zotero' mode, use ONLY these references as your evidence base.")
        lines.append("- Consider software-specific, method-specific, and pair-specific items when judging implementation.")
        lines.append("- Rank: 0–3, as defined in the system instructions.")
        return "\n".join(lines)


    # ---------- LLM backends (OpenAI, Claude, Gemini, Perplexity/Sonar) ----------

    def _call_openai(self, prompt: str, model: str) -> Tuple[str, int, int]:
        messages = [
            {"role": "user", "content": prompt}
        ]
        input_tokens = count_tokens_in_messages(messages, model)
        for attempt in range(self.max_retries):
            try:
                resp = openai_client.chat.completions.create(
                    model=model,
                    messages=messages,
                    temperature=0.2,
                    max_tokens=4096,
                    timeout=self.timeout,
                    response_format={"type": "json_object"}  # minimize parsing issues
                )
                content = resp.choices[0].message.content
                usage = resp.usage
                output_tokens = usage.completion_tokens
                credit_tracker.update(model, input_tokens, output_tokens)
                return content, input_tokens, output_tokens
            except Exception as e:
                print(f"[OpenAI] Error attempt {attempt+1}: {e}")
                time.sleep(2)
        raise RuntimeError("OpenAI call failed after retries")

    def _call_claude(self, prompt: str, model: str) -> Tuple[str, int, int]:
        if anthropic_client is None:
            raise RuntimeError("Anthropic client not initialized")
        for attempt in range(self.max_retries):
            try:
                resp = anthropic_client.messages.create(
                    model=model,
                    max_tokens=4096,
                    temperature=0.2,
                    system="Return ONLY JSON.",
                    messages=[{"role": "user", "content": prompt}],
                    timeout=self.timeout,
                )
                content = resp.content[0].text
                usage = resp.usage
                input_tokens = usage.input_tokens
                output_tokens = usage.output_tokens
                credit_tracker.update(model, input_tokens, output_tokens)
                return content, input_tokens, output_tokens
            except Exception as e:
                print(f"[Claude] Error attempt {attempt+1}: {e}")
                time.sleep(2)
        raise RuntimeError("Claude call failed after retries")

    def _call_gemini(self, prompt: str, model: str) -> Tuple[str, int, int]:
        if not google_enabled:
            raise RuntimeError("Google Gemini not enabled")
        for attempt in range(self.max_retries):
            try:
                m = genai.GenerativeModel(model)
                resp = m.generate_content(prompt, generation_config={"temperature": 0.2})
                content = resp.text
                # Gemini usage is not as explicit; approximate tokens
                input_tokens = len(prompt.split()) // 0.75
                output_tokens = len(content.split()) // 0.75
                credit_tracker.update(model, int(input_tokens), int(output_tokens))
                return content, int(input_tokens), int(output_tokens)
            except Exception as e:
                print(f"[Gemini] Error attempt {attempt+1}: {e}")
                time.sleep(2)
        raise RuntimeError("Gemini call failed after retries")

    def _call_perplexity(self, prompt: str, model: str) -> Tuple[str, int, int]:
        if perplexity_client is None:
            raise RuntimeError("Perplexity client not initialized")
        messages = [
            {"role": "user", "content": prompt}
        ]
        input_tokens = count_tokens_in_messages(messages, model)
        for attempt in range(self.max_retries):
            try:
                resp = perplexity_client.chat.completions.create(
                    model=model,
                    messages=messages,
                    temperature=0.3,
                    max_tokens=4096,
                    timeout=self.timeout,
                )
                content = resp.choices[0].message.content
                usage = resp.usage
                output_tokens = usage.completion_tokens
                credit_tracker.update(model, input_tokens, output_tokens)
                return content, input_tokens, output_tokens
            except Exception as e:
                print(f"[Perplexity] Error attempt {attempt+1}: {e}")
                time.sleep(3)
        raise RuntimeError("Perplexity call failed after retries")

    # ---------- Batch assessment for a single backend ----------

    def assess_batch(self,
                     items: List[Tuple[str, str]],
                     provider: str,
                     model: str,
                     mode: str,
                     batch_id: str,
                     zotero_refs: Optional[Dict[Tuple[str, str], List[Dict]]] = None
                     ) -> List[AssessmentResult]:
        """
        provider: "openai" | "claude" | "gemini" | "perplexity"
        mode: "zotero" | "web"
        """
        prompt = self.build_batch_prompt(items, mode, zotero_refs=zotero_refs)

        if provider == "openai":
            raw, in_tok, out_tok = self._call_openai(prompt, model)
        elif provider == "claude":
            raw, in_tok, out_tok = self._call_claude(prompt, model)
        elif provider == "gemini":
            raw, in_tok, out_tok = self._call_gemini(prompt, model)
        elif provider == "perplexity":
            raw, in_tok, out_tok = self._call_perplexity(prompt, model)
        else:
            raise ValueError(f"Unknown provider {provider}")

        # Parse JSON
        try:
            parsed = json.loads(raw)
        except Exception:
            # Claude/OpenAI with json_object should return dict; accept dict["results"]
            try:
                cleaned = raw.strip()
                if cleaned.startswith("```"):
                    cleaned = "\n".join(
                        line for line in cleaned.splitlines()
                        if not line.strip().startswith("```")
                    )
                parsed = json.loads(cleaned)
            except Exception as e:
                print("Failed to parse LLM JSON:", e)
                return []

        if isinstance(parsed, dict):
            # if wrapped
            if "results" in parsed:
                parsed = parsed["results"]
            else:
                # assume dict of lists
                parsed = next((v for v in parsed.values() if isinstance(v, list)), [])

        results: List[AssessmentResult] = []
        per_item_in = in_tok // max(len(items), 1)
        per_item_out = out_tok // max(len(items), 1)

        for obj in parsed:
            sw = obj.get("software", "").strip()
            meth = obj.get("method", "").strip()
            rank = int(obj.get("rank", 0))
            reasoning = obj.get("reasoning", "")
            sources = obj.get("sources", [])
            if isinstance(sources, str):
                sources = [sources]
            results.append(
                AssessmentResult(
                    software=sw,
                    method=meth,
                    rank=rank,
                    reasoning=reasoning,
                    sources=sources,
                    llm_provider=provider,
                    llm_model=model,
                    mode=mode,
                    batch_id=batch_id,
                    input_tokens=per_item_in,
                    output_tokens=per_item_out,
                )
            )
        return results

    # ---------- Consensus aggregation ----------

    def aggregate_consensus(self,
                            assessments: List[AssessmentResult]
                            ) -> List[ConsensusResult]:
        """
        Group by (software, method) and compute:
        - final_rank (majority)
        - confidence + agreement_level
        - individual_ranks, modes, sources
        """
        grouped: Dict[Tuple[str, str], List[AssessmentResult]] = defaultdict(list)
        for r in assessments:
            grouped[(r.software, r.method)].append(r)

        consensus: List[ConsensusResult] = []
        for (sw, meth), group in grouped.items():
            ranks = [g.rank for g in group]
            conf, level = self.calc_confidence(ranks)

            # majority rank
            c = Counter(ranks)
            final_rank = c.most_common(1)[0][0]

            indiv_ranks = {}
            indiv_modes = {}
            indiv_sources = {}
            for g in group:
                key = f"{g.llm_provider}:{g.llm_model}:{g.batch_id}"
                indiv_ranks[key] = g.rank
                indiv_modes[key] = g.mode
                indiv_sources[key] = g.sources

            consensus.append(
                ConsensusResult(
                    software=sw,
                    method=meth,
                    final_rank=final_rank,
                    confidence=conf,
                    agreement_level=level,
                    individual_ranks=indiv_ranks,
                    individual_modes=indiv_modes,
                    individual_sources=indiv_sources,
                    total_assessments=len(group),
                )
            )
        return consensus


In [ ]:
# Cell 4: Loading lists, Zotero references, existing results

def load_software_list(csv_path: Path,
                       name_col: str = "Name",
                       filter_col: Optional[str] = None,
                       filter_value: Optional[str] = None) -> List[str]:
    df = pd.read_csv(csv_path, sep=";", encoding="latin1")
    if filter_col and filter_value is not None and filter_col in df.columns:
        df = df[df[filter_col] == filter_value]
    return df[name_col].dropna().tolist()

def load_method_list(variant_json: Path) -> List[str]:
    with open(variant_json, "r", encoding="utf-8") as f:
        groups = json.load(f)
    return list(groups.keys())

def load_zotero_refs(mapping_file: Optional[Path] = None
                     ) -> Dict[str, Dict]:
    """
    Returns a dict with three maps:
      - by_software[software] -> [items]  (software-tagged)
      - by_method[method] -> [items]      (method-tagged)
      - by_pair[(software, method)] -> [items]  (tagged with both)
    """
    if mapping_file is None or not mapping_file.exists():
        return {
            "by_software": defaultdict(list),
            "by_method": defaultdict(list),
            "by_pair": defaultdict(list),
        }

    with open(mapping_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    by_software: Dict[str, List[Dict]] = defaultdict(list)
    by_method: Dict[str, List[Dict]] = defaultdict(list)
    by_pair: Dict[Tuple[str, str], List[Dict]] = defaultdict(list)

    for r in data:
        sw = r.get("software", "").strip()
        meth = r.get("method", "").strip()

        if sw:
            by_software[sw].append(r)
        if meth:
            by_method[meth].append(r)
        if sw and meth:
            by_pair[(sw, meth)].append(r)

    return {
        "by_software": by_software,
        "by_method": by_method,
        "by_pair": by_pair,
    }

def load_existing_assessments(path: Path) -> List[AssessmentResult]:
    if not path.exists():
        return []
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    results = []
    for obj in data:
        results.append(AssessmentResult(**obj))
    return results

def save_assessments(path: Path, results: List[AssessmentResult]) -> None:
    serial = [asdict(r) for r in results]
    with open(path, "w", encoding="utf-8") as f:
        json.dump(serial, f, indent=2, ensure_ascii=False)

def load_existing_consensus(path: Path) -> List[ConsensusResult]:
    if not path.exists():
        return []
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    results = []
    for obj in data:
        results.append(ConsensusResult(**obj))
    return results

def save_consensus(path: Path, results: List[ConsensusResult]) -> None:
    serial = [asdict(r) for r in results]
    with open(path, "w", encoding="utf-8") as f:
        json.dump(serial, f, indent=2, ensure_ascii=False)


In [ ]:
# Cell 5: End-to-end run (fresh or incremental) and matrix creation

from datetime import datetime

# System prompts specialized for Zotero vs web
SYSTEM_PROMPT_ZOTERO = """You are a technical software assessment expert specialized in power systems analysis software.
Use this ranking scale:
0: No support; method cannot be implemented at all
1: Limited possibility; requires major workarounds
2: Indirectly supported via APIs, plugins, or external tools
3: Directly implemented as a native feature.

You must base your assessment ONLY on the references provided for each software–method pair
(e.g., Zotero items given in the prompt). Do not invent or search for additional references.

Return JSON with fields: software, method, rank (0–3), reasoning, sources (list of strings)."""

SYSTEM_PROMPT_WEB = """You are a technical software assessment expert specialized in power systems analysis software.
Use this ranking scale:
0: No support; method cannot be implemented at all
1: Limited possibility; requires major workarounds
2: Indirectly supported via APIs, plugins, or external tools
3: Directly implemented as a native feature.

For each software–method pair, you may search the web and scientific literature
(official documentation, peer-reviewed papers, open-source implementations, etc.).
Cite only real, verifiable URLs in 'sources'.

Return JSON with fields: software, method, rank (0–3), reasoning, sources (list of strings)."""

assessor = SoftwareMethodAssessor(
    system_prompt_zotero=SYSTEM_PROMPT_ZOTERO,
    system_prompt_web=SYSTEM_PROMPT_WEB,
)

# LLM configurations you want to rotate between
# Example: two Zotero-constrained models + two web-search models
ZOTERO_LLM_CONFIG = [
    ("openai", default_openai_model),
    ("claude", "claude-3-5-haiku-20241022"),
]

WEB_LLM_CONFIG = [
    ("perplexity", "sonar"),                 # Perplexity web search
    ("openai", default_openai_model),        # OpenAI unconstrained (but prompted to use web)
]

# ---- Load software, methods, Zotero references, and existing results ----
software_list = load_software_list(SOFTWARE_CSV, name_col="Name",
                                   filter_col=None, filter_value=None)
method_list = load_method_list(METHOD_VARIANTS_JSON)

# Construct all pairs
all_pairs = [(s, m) for s in software_list for m in method_list]

# Zotero mapping (optional)
ZOTERO_MAPPING_FILE = DATA_DIR / "zotero_pair_refs.json"
zotero_refs = load_zotero_refs(ZOTERO_MAPPING_FILE)

# Existing assessments (if resuming)
existing_assessments = load_existing_assessments(ASSESS_RAW_JSON)
done_pairs = {(r.software, r.method, r.mode, r.llm_provider, r.llm_model) for r in existing_assessments}

print(f"Loaded {len(existing_assessments)} existing assessment records.")

# ---- Helper to select remaining items for a given mode + (provider, model) ----
def remaining_pairs_for_config(all_pairs: List[Tuple[str, str]],
                               existing: List[AssessmentResult],
                               mode: str,
                               provider: str,
                               model: str) -> List[Tuple[str, str]]:
    done = {(r.software, r.method) for r in existing
            if r.mode == mode and r.llm_provider == provider and r.llm_model == model}
    return [p for p in all_pairs if p not in done]




In [ ]:
# ---- Main loop: run in batches ----
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
batch_counter = 0

for mode, llm_config in [("zotero", ZOTERO_LLM_CONFIG), ("web", WEB_LLM_CONFIG)]:
    print(f"\n=== MODE: {mode} ===")
    for provider, model in llm_config:
        remaining = remaining_pairs_for_config(all_pairs, existing_assessments,
                                               mode=mode, provider=provider, model=model)
        print(f"Provider {provider}/{model}: {len(remaining)} pairs remaining.")
        if not remaining:
            continue

        for i in range(0, len(remaining), BATCH_SIZE):
            batch_items = remaining[i:i + BATCH_SIZE]
            batch_counter += 1
            batch_id = f"{timestamp}_b{batch_counter:04d}"
            print(f"  Batch {batch_id}: {len(batch_items)} pairs")

            try:
                batch_results = assessor.assess_batch(
                    items=batch_items,
                    provider=provider,
                    model=model,
                    mode=mode,
                    batch_id=batch_id,
                    zotero_refs=zotero_refs if mode == "zotero" else None,
                )
            except Exception as e:
                print(f"Batch {batch_id} failed: {e}")
                continue

            existing_assessments.extend(batch_results)
            save_assessments(ASSESS_RAW_JSON, existing_assessments)
            print(f"  -> total stored assessments: {len(existing_assessments)}")

print("\nAssessments complete.")
print(credit_tracker.get_summary_text())

In [ ]:
# Cell 6: Build consensus and write matrix CSV

# Load all raw assessments
all_assessments = load_existing_assessments(ASSESS_RAW_JSON)
print(f"Total assessments loaded: {len(all_assessments)}")

# Aggregate consensus
consensus_results = assessor.aggregate_consensus(all_assessments)
save_consensus(ASSESS_AGG_JSON, consensus_results)
print(f"Consensus entries: {len(consensus_results)}")

# Build matrix (software x method) from consensus
# Final rank can be average across modes/models; here majority rank already computed.
software_set = sorted({c.software for c in consensus_results})
method_set = sorted({c.method for c in consensus_results})

# Initialize matrix with NaN
mat = pd.DataFrame(index=software_set, columns=method_set, dtype=float)

# For each (software, method), put final_rank; optionally weight by confidence
for c in consensus_results:
    mat.at[c.software, c.method] = c.final_rank

mat.index.name = "Software"
mat.reset_index(inplace=True)
mat.to_csv(MATRIX_CSV, sep=";", index=False, encoding="utf-8-sig")
print(f"Matrix written to {MATRIX_CSV}")


In [ ]:
# Cell 7: Quick diagnostics

consensus_results = load_existing_consensus(ASSESS_AGG_JSON)
dfc = pd.DataFrame([asdict(c) for c in consensus_results])

print("Confidence distribution:")
print(dfc["confidence"].describe())

print("\nAgreement levels:")
print(dfc["agreement_level"].value_counts())

low_conf = dfc[dfc["confidence"] < 0.75]
print(f"\nPairs with confidence < 0.75: {len(low_conf)}")
low_conf.head(10)
